# Hyperparameter Sweep (Optuna) - PatchCore

This notebook coordinates the hyperparameter optimization sweep across all 15 MVTec AD categories for PatchCore.

**Memory Protection**: Running extensive PatchCore trials inside a single long-lived Jupyter kernel can lead to CUDA memory fragmentation. We provide two execution options:
1. **Subprocess Runner (Recommended)**: `!python scripts/sweep.py --model patchcore` runs trials in isolated subprocesses, preventing memory accumulation.
2. **In-Notebook Interactive Loop**: Iterates through categories with aggressive cache clearing (`torch.cuda.empty_cache()` and `gc.collect()`), preloading CUDA 12 shared libraries and persisting checkpoints incrementally.

In [ ]:
import os
import sys
import warnings
from pathlib import Path

# Suppress noisy deprecation warnings from external libraries
warnings.filterwarnings("ignore", category=FutureWarning, module=".*timm.*")
warnings.filterwarnings("ignore", category=FutureWarning, module=".*anomalib.*")

# Dynamically navigate to repository root (supports both local environment and Google Colab)
current_dir = Path.cwd()
while not (current_dir / "app").exists() and current_dir != current_dir.parent:
    current_dir = current_dir.parent
PROJECT_ROOT = current_dir
os.chdir(PROJECT_ROOT)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from app.core.tf_device import preload_cuda_shared_libraries
preload_cuda_shared_libraries()

print(f"Project root resolved: {PROJECT_ROOT.resolve()}")


In [ ]:
# Option A (Recommended): Run PatchCore sweep via isolated subprocesses to prevent CUDA memory leaks
!python scripts/sweep.py --model patchcore


In [ ]:
# Option B: Run in-notebook interactive loop with incremental saving and cache clearing
import gc
import json
import torch
from app.pipelines.modelling.patchcore.optuna_study import run_study as run_patchcore_study

DATA_ROOT = str(PROJECT_ROOT / "data/raw/mvtec_ad")
CATEGORIES = [
    "bottle", "cable", "capsule", "carpet", "grid",
    "hazelnut", "leather", "metal_nut", "pill", "screw",
    "tile", "toothbrush", "transistor", "wood", "zipper",
]

patchcore_output = PROJECT_ROOT / "data/hyperparameters/patchcore_best.json"
patchcore_output.parent.mkdir(parents=True, exist_ok=True)

if patchcore_output.exists():
    patchcore_results = json.loads(patchcore_output.read_text(encoding="utf-8"))
    print(f"Loaded existing progress ({len(patchcore_results)} completed): {list(patchcore_results.keys())}")
else:
    patchcore_results = {}

for category in CATEGORIES:
    if category in patchcore_results:
        continue

    print(f"\n{'=' * 50}\nRunning Optuna study for: {category}\n{'=' * 50}\n--- PatchCore ---")
    patchcore_cfg = run_patchcore_study(category_name=category, n_trials=30, data_root=DATA_ROOT)
    patchcore_results[category] = patchcore_cfg

    # Save incrementally after each category to prevent progress loss
    patchcore_output.write_text(json.dumps(patchcore_results, indent=2))
    print(f"[INFO] Successfully saved {category} progress to {patchcore_output}")

    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    gc.collect()

print(f"\nPatchCore sweep complete: {patchcore_output.resolve()}")


In [ ]:
import json

# Inspect saved PatchCore hyperparameter configurations
results_path = PROJECT_ROOT / "data/hyperparameters/patchcore_best.json"
if results_path.exists():
    results = json.loads(results_path.read_text(encoding="utf-8"))
    print(f"Completed categories ({len(results)}/15): {list(results.keys())}\n")
    print(json.dumps(results, indent=2))
else:
    print(f"No hyperparameter file found at {results_path}")
